In [3]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
import utils_2Q_gate_zp as ut
ut.set_fig_font() ### Set various sizes in plotting
import scipy as sp
from joblib import Parallel, delayed
import itertools
from qutip.qip.operations import rz, cz_gate, cnot, rx, hadamard_transform, swap
import pandas as pd

### Optimize fidelity over tg

In [4]:
truc1, truc_tot, charge_pick = 300, 1000, True
truc_tot_2 = 500

folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta0_dress.txt').to_numpy()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
truc_list = np.arange(truc_tot_2)
hspace_full = hspace_full[:truc_tot_2]
eval_tot = eval_tot[:truc_tot_2]
n_theta0_dress = ut.truncate_2(n_theta0_dress, truc_list)

A1_bound = (0, 0.4)
tg_bound = (-1, 1) #(-1e-10, 1e-10) #
tg_vec = [200]#np.arange(20, 101, 10)

workers, popsize = 100, 10
recombination, tol, mutation = [0.7, 0.01, (0.5, 1.0)]

logi_state = ['0-0', '0-2', '2-0', '2-2']
idx_0 = hspace_full.index('0-2')
idx_1 = hspace_full.index('2-2')
idx_2 = hspace_full.index('8-2')
W_0_2 = eval_tot[idx_2] - eval_tot[idx_0]
W_1_2 = eval_tot[idx_2] - eval_tot[idx_1]
n_theta0_0_2 = n_theta0_dress[idx_2, idx_0]
n_theta0_1_2 = n_theta0_dress[idx_2, idx_1]

# hspace_truc = hspace_full[:100]
hspace_truc = [
'0-0', '0-2', '2-2', '8-2', '2-0', '12-2', '1-2', '1-0', '5-2', '5-0' ,
'4-9', '8-5', '2-1', '22-2', '0-1', '9-2', '5-8', '2-5', '5-5', '34-2' ,
'8-0', '0-5', '13-2', '1-4', '30-2', '20-5', '4-2', '15-5', '15-0', '15-2' ,
'1-1', '8-9', '26-5', '24-2', '26-2', '9-0', '20-2', '25-2', '4-5', '4-0' ,
'5-1', '12-5', '12-0', '18-2', '9-5', '1-5', '13-0', '37-2', '35-2', '9-4' ,
'8-1', '20-0', '4-1', '9-1', '13-9', '33-2', '1-13', '18-0', '18-5', '9-9' ,
'12-8', '5-4', '0-21', '9-8', '1-8', '26-0', '5-12', '5-16', '15-1', '22-5' ,
'25-4', '1-25', '4-4', '5-9', '2-4', '18-4', '9-12', '15-4', '1-18', '64-2' ,
'45-0', '12-9', '13-5', '4-8', '44-2', '8-4', '45-5', '25-5', '2-12', '12-1' ,
'0-8', '50-2', '4-12', '45-2', '2-21', '18-1', '15-9', '25-0', '1-9', '2-8' ,
]
truc_index = [hspace_full.index(i) for i in hspace_truc]
truc_len = len(hspace_truc)

H0_full = qt.Qobj(np.diag(eval_tot))
logi_idx_full = [hspace_full.index(i) for i in logi_state]
H_drive_full = [ H0_full,   [n_theta0_dress, ut.drive_gauss_A],
                            [n_theta0_dress, ut.drive_gauss_B]  ]

H0_truc = ut.truncate_2( H0_full, truc_index)
n_theta0_truc = ut.truncate_2(n_theta0_dress, truc_index)
logi_idx_truc = [hspace_truc.index(i) for i in logi_state]
H_drive_truc = [ H0_truc,   [n_theta0_truc, ut.drive_gauss_A],
                            [n_theta0_truc, ut.drive_gauss_B]  ]

c_op_list = []
num_cpus = None
def cnot_fidelity_log_tg_only(arg_optimize, *args):
    tg, drive_amp_A = arg_optimize
    drive_amp_B = drive_amp_A * n_theta0_0_2 / n_theta0_1_2

    [H_qbt_drive, w_0_2, w_1_2, num_cpus, c_op_list, logi_idx] = args
    detune_A, detune_B = 0, 0

    arg_all = [tg, drive_amp_A, drive_amp_B, detune_A, detune_B,
     H_qbt_drive, w_0_2, w_1_2, num_cpus, c_op_list, logi_idx]
    return ut.cnot_fidelity_log(arg_all)

In [6]:
print(hspace_full.index('8-2'))
print(hspace_full.index('5-0'))

37
10


In [ ]:
fidelity = []
drive_param = []
fidelity_full = []
arg_truc = [H_drive_truc, W_0_2, W_1_2, num_cpus, c_op_list, logi_idx_truc]
for jdx, tg in tqdm(enumerate(tg_vec)):
    tg_bounds = (tg+tg_bound[0], tg+tg_bound[1])
    bounds = (tg_bounds, A1_bound)

    res = sp.optimize.differential_evolution(
        func=cnot_fidelity_log_tg_only,
        bounds=bounds,
        args=arg_truc,
        disp=True,
        callback=ut.print_soln,
        init="sobol",
        workers=workers,
        popsize=popsize,
        mutation=mutation,
        recombination=recombination,
        tol=tol,
        polish=False, # 'True' will make the for-loop break
        )
    fidelity.append(res.fun)
    drive_param.append(res.x.tolist())
    print(res, '\n')
    print('\ntg = ', np.array(tg_vec[:jdx+1]).tolist())

    print(f'\nlog of gate error (truc={truc_len}) = ')
    for i in range(0, len(fidelity), 4):
        print(', '.join(map(str, np.round(fidelity[i:i+4], 8))), ',')

    print(f'\ndrive_param (truc={truc_len}) = ')
    for i in drive_param:
        print(np.round(i,6).tolist(),',')

0it [00:00, ?it/s]UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
 /home/zlqed/anaconda3/envs/qutip4/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py: 387

differential_evolution step 1: f(x)= -0.170544
Best Solution: [199.5881766, 0.11963657]
Convergence: 0.0576
----------------------------
differential_evolution step 2: f(x)= -0.170544
Best Solution: [199.5881766, 0.11963657]
Convergence: 0.0646
----------------------------
differential_evolution step 3: f(x)= -0.172489
Best Solution: [200.24765092, 0.11859568]
Convergence: 0.0653
----------------------------
differential_evolution step 4: f(x)= -0.172489
Best Solution: [200.24765092, 0.11859568]
Convergence: 0.0752
----------------------------
differential_evolution step 5: f(x)= -0.178062
Best Solution: [200.59142451, 0.11963657]
Convergence: 0.0766
----------------------------
differential_evolution step 6: f(x)= -0.178663
Best Solution: [199.9406455, 0.15275048]
Convergence: 0.0931
----------------------------
differential_evolution step 7: f(x)= -0.178663
Best Solution: [199.9406455, 0.15275048]
Convergence: 0.1
----------------------------
differential_evolution step 8: f(x)= -0.2

### Get fidelity for input params

In [ ]:
truc1, truc_tot, charge_pick = 300, 1000, True
truc_tot_2 = 400

folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta0_dress.txt').to_numpy()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
truc_list = np.arange(truc_tot_2)
hspace_full = hspace_full[:truc_tot_2]
eval_tot = eval_tot[:truc_tot_2]
n_theta0_dress = ut.truncate_2(n_theta0_dress, truc_list)

workers, popsize = 100, 10
recombination, tol, mutation = [0.7, 0.01, (0.5, 1.0)]

logi_state = ['0-0', '0-2', '2-0', '2-2']
idx_0 = hspace_full.index('0-2')
idx_1 = hspace_full.index('2-2')
idx_2 = hspace_full.index('8-2')
W_0_2 = eval_tot[idx_2] - eval_tot[idx_0]
W_1_2 = eval_tot[idx_2] - eval_tot[idx_1]
n_theta0_0_2 = n_theta0_dress[idx_2, idx_0]
n_theta0_1_2 = n_theta0_dress[idx_2, idx_1]

num_cpus = None
x0_vec = np.array([
[19.991937, 0.079679, 0.151403, 0.068109, 0.115777] ,
[100.009005, 0.10977, 0.149046, 0.078162, 0.158805] ,
])
n_job = 100
c_op_list = []

H0_full = qt.Qobj(np.diag(eval_tot))
logi_idx_full = [hspace_full.index(i) for i in logi_state]
H_drive_full = [ H0_full,   [n_theta0_dress, ut.drive_gauss_A],
                            [n_theta0_dress, ut.drive_gauss_B]  ]

In [ ]:
arg_full = [H_drive_full, W_0_2, W_1_2, num_cpus, c_op_list, logi_idx_full]
f_full = Parallel(n_jobs=n_job)(delayed(ut.cnot_fidelity_log_tg)(args_indep, *arg_full)
                                            for args_indep in x0_vec)
print(f' f_full (dim={truc_tot_2}) = [')
for i in range(0, len(f_full), 4):
    print(', '.join(map(str, f_full[i:i+4])), ',')
print(']')

f_full (dim=400) = [
-0.1306422833144859, -0.1854447506209772 ,
]


In [ ]:
hspace_truc = hspace_full[:300]
truc_index = [hspace_full.index(i) for i in hspace_truc]
truc_len = len(hspace_truc)
H0_truc = ut.truncate_2( H0_full, truc_index)
n_theta0_truc = ut.truncate_2(n_theta0_dress, truc_index)
logi_idx_truc = [hspace_truc.index(i) for i in logi_state]
H_drive_truc = [ H0_truc,   [n_theta0_truc, ut.drive_gauss_A],
                            [n_theta0_truc, ut.drive_gauss_B]  ]

arg_truc = [H_drive_truc, W_0_2, W_1_2, num_cpus, c_op_list, logi_idx_truc]
f_truc = Parallel(n_jobs=n_job)(delayed(ut.cnot_fidelity_log_tg)(args_indep, *arg_truc)
                                            for args_indep in x0_vec)
print(f' f_truc (dim={len(hspace_truc)}) = [')
for i in range(0, len(f_truc), 4):
    print(', '.join(map(str, f_truc[i:i+4])), ',')
print(']')


f_truc (dim=300) = [
-0.16491349218022386, -0.17966493702367645 ,
]


In [ ]:
hspace_select = [
'0-0', '0-1', '1-0', '0-2', '2-0', '4-0', '1-1', '0-5', '2-1', '5-0' ,
'1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '9-0', '5-1', '4-2', '2-5' ,
'1-8', '12-0', '5-2', '8-1', '4-4', '13-0', '15-0', '2-8', '1-9', '9-1' ,
'8-2', '5-4', '4-5', '0-21', '18-0', '9-2', '5-5', '4-8', '1-13', '20-0' ,
'22-0', '2-12', '15-1', '5-8', '4-9', '12-2', '9-4', '8-5', '25-0', '13-2' ,
'1-18', '15-2', '5-9', '4-12', '18-1', '12-4', '9-5', '1-21', '8-8', '20-1' ,
'4-13', '30-0', '13-4', '18-2', '34-0', '8-9', '12-5', '20-2', '5-16', '22-2' ,
'24-2', '15-5', '9-9', '12-8', '0-45', '25-2', '26-2', '9-12', '12-9', '28-2' ,
'18-5', '46-0', '13-9', '9-13', '20-5', '30-2', '25-4', '8-18', '33-2', '18-8' ,
'34-2', '35-2', '37-2', '26-5', '24-9', '30-5', '44-2', '35-5', '37-5', '56-2' ,
]
index_select = [hspace_full.index(i) for i in hspace_select]
len_select = len(hspace_select)
H0_select = ut.truncate_2( H0_full, index_select)
n_theta0_select = ut.truncate_2(n_theta0_dress, index_select)
logi_idx_select = [hspace_select.index(i) for i in logi_state]
H_drive_select = [ H0_select,   [n_theta0_select, ut.drive_gauss_A],
                                [n_theta0_select, ut.drive_gauss_B]  ]

arg_select = [H_drive_select, W_0_2, W_1_2, num_cpus, c_op_list, logi_idx_select]
f_select = Parallel(n_jobs=n_job)(delayed(ut.cnot_fidelity_log_tg)(args_indep, *arg_select)
                                            for args_indep in x0_vec)
print(f' f_select_normalize (dim={len(hspace_select)}) = [')
for i in range(0, len(f_select), 4):
    print(', '.join(map(str, f_select[i:i+4])), ',')
print(']')

f_select_normalize (dim=100) = [
-0.38721393819099414, -0.65727601066842 ,
]


In [ ]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [ ]:
c_op_list = []
num_cpus = None
arg_truc = [H_drive_truc, W_00_80, W_20_80, num_cpus, c_op_list, logi_idx_truc]


[tg, drive_amp_A, drive_amp_B, detune_A, detune_B] = 10.108462,0.41212,0.073451,-0.084451,-0.044706
[H_qbt_drive, w_0_2, w_1_2, num_cpus, c_op_list, logi_idx] = [H_drive_truc, W_00_80, W_20_80, num_cpus, c_op_list, logi_idx_truc]

pulse_args = {'drive_amp_A': drive_amp_A ,
            'drive_freq_A': w_0_2 + 2*np.pi*detune_A,
            'drive_amp_B': drive_amp_B ,
            'drive_freq_B': w_1_2 + 2*np.pi*detune_B,
            'gate_time': tg }
tlist = np.linspace(0, tg, num=int(tg))  # total time
U_noise = ut.get_propagator(H_qbt_drive, tlist, num_cpus, c_op_list, pulse_args, logi_idx)
p0_kraus = qt.to_kraus(qt.to_super(U_noise))
if len(c_op_list) != 0:
    p0_kraus = [ut.truncate_2(i, logi_idx) for i in p0_kraus]
p0_kraus_zz = ut.cnot_phase_correct(p0_kraus)
p0_super_2 = qt.kraus_to_super(p0_kraus_zz)
print(np.shape(p0_kraus))
print(np.shape(p0_kraus_zz))
print(np.shape(p0_super_2))
f_noise = qt.metrics.average_gate_fidelity(p0_super_2, target=qt.Qobj(cnot().full()))

(1, 4, 4)
(1, 4, 4)
(16, 16)
